In [1]:
import pandas as pd

df_movies = pd.read_csv("..\\datasets\\movies.dat",
                        sep='::', engine='python',
                        encoding="Latin-1",
                        names=['movie_id', 'title', 'genre'])

df_rating = pd.read_csv("..\\datasets\\ratings.dat",
                        sep='::', engine='python',
                        encoding="Latin-1",
                        names=['user_id', 'movie_id', 'rating', 'timestamp'])

df_users = pd.read_csv("..\\datasets\\users.dat",
                       sep='::', engine='python',
                       encoding="Latin-1",
                       names=["user_id", "gender", "age_group", "occupation", "zip_code"])

In [2]:
df_movies.info()
df_movies_processed = df_movies.astype({
    "movie_id": "int32", 
    "title": "string",
    "genre": "string"
})

<class 'pandas.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  3883 non-null   int64
 1   title     3883 non-null   str  
 2   genre     3883 non-null   str  
dtypes: int64(1), str(2)
memory usage: 225.4 KB


In [3]:
df_rating.info()
df_rating_processed = df_rating.astype({
    "user_id": "int32",
    "movie_id": "int32",
    "rating": "float32",
    "timestamp": "int32"
})

<class 'pandas.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1000209 non-null  int64
 1   movie_id   1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [4]:
df_users.info()
df_users_processed = df_users.astype({
    "user_id": "int32",
    "age_group": "int8",
    "occupation": "int8",
    "zip_code": "string"
})

<class 'pandas.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     6040 non-null   int64
 1   gender      6040 non-null   str  
 2   age_group   6040 non-null   int64
 3   occupation  6040 non-null   int64
 4   zip_code    6040 non-null   str  
dtypes: int64(3), str(2)
memory usage: 271.8 KB


### Movies

In [5]:
genres = (
    df_movies_processed.genre.str.get_dummies('|')
)

df_movies_processed = pd.concat(
    [
        df_movies_processed,
        genres
    ],
    axis=1
)

df_movies_processed.head(10)

,movie_id,title,genre,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Animation|Children's|Comedy,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,6,Heat (1995),Action|Crime|Thriller,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
6,7,Sabrina (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
7,8,Tom and Huck (1995),Adventure|Children's,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,9,Sudden Death (1995),Action,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,10,GoldenEye (1995),Action|Adventure|Thriller,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


### Rating


In [6]:
from sklearn.preprocessing import LabelEncoder

In [7]:
df_interactions = df_rating_processed.copy()
df_interactions = df_interactions.drop(columns=["timestamp"])

user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

df_interactions["user_idx"] = (
    user_encoder.fit_transform(df_interactions["user_id"])
)

df_interactions["movie_idx"] = (
    movie_encoder.fit_transform(df_interactions["movie_id"]) 
)

df_interactions.head(10)

,user_id,movie_id,rating,user_idx,movie_idx
0,1,1193,5.0,0,1104
1,1,661,3.0,0,639
2,1,914,3.0,0,853
3,1,3408,4.0,0,3177
4,1,2355,5.0,0,2162
5,1,1197,3.0,0,1107
6,1,1287,5.0,0,1195
7,1,2804,5.0,0,2599
8,1,594,4.0,0,580
9,1,919,4.0,0,858


In [8]:
n_users = df_interactions["user_idx"].nunique()
n_movies = df_interactions["movie_idx"].nunique()

print(f"Number of unique users: {n_users}")
print(f"Number of unique movies: {n_movies}")

Number of unique users: 6040
Number of unique movies: 3706


### User

In [9]:
df_users_processed['gender'] = (
    df_users_processed['gender']
    .str.strip()
    .map({'F': 0, 'M': 1})
    .astype("int8")
)
df_users_processed.head(10)

,user_id,gender,age_group,occupation,zip_code
0,1,0,1,10,48067
1,2,1,56,16,70072
2,3,1,25,15,55117
3,4,1,45,7,02460
4,5,1,25,20,55455
5,6,0,50,9,55117
6,7,1,35,1,06810
7,8,1,25,12,11413
8,9,1,25,17,61614
9,10,0,35,1,95370


### Save preprocessed dateframes

In [10]:
import joblib

joblib.dump(
    {
        "user_encoder" : user_encoder,
        "movie_encoder": movie_encoder
    },
    "data\\encoders.pkl"
)

joblib.dump(
    {
        "n_movies": n_movies,
        "n_users": n_users
    },
    "data\\metadata.pkl"
)

df_movies_processed.to_parquet(
    "data\\movies.parquet",
    index=False
)

df_interactions[["user_idx", "movie_idx", "rating"]].to_parquet(
    "data\\interactions.parquet",
    index=False
)

df_users_processed.to_parquet(
    "data\\users.parquet",
    index=False
)